# (Complex-Scaled) Restricted Hartree Fock
This notebook serves as a brief guide of the results obtained, explanation of the use and so on. We use these external libraries:

In [1]:
import numpy as np 
from pathlib import Path
import matplotlib.pyplot as plt 
from pyscf import gto, scf

## RHF implementation
RHF has been implemented as sen in Szabo. Currently the RHF routine requires to input the value of molecular integrals: $T$, $V$, $S$, and $(ij|kl)$. The implementation can be easily compared with known results as the $H_2$ example in Szabo and the one obtained using using Pyscf. In the $H_2$ case, the total HF energy obtained for two $H$ atoms separated 1.4 atomic units is:
$$
E_{HF}^{sto3g} = -1.1167
$$
Inclyding the $V_{NN}$ terms. The same calculation using Pyscf can be performed as:

In [2]:
dist = 1.4 * 0.529177249

mol_H2 = gto.M(atom = f'H 0 0 0; H 0 0 {dist}', spin=0)

T_sto3g_H2 = mol_H2.intor('int1e_kin')
V_sto3g_H2 = mol_H2.intor('int1e_nuc')
S_sto3g_H2 = mol_H2.intor('int1e_ovlp')
eri_sto3g_H2 = mol_H2.intor('int2e')

rhf_H2 = scf.RHF(mol_H2)

pyscf_e_H2 = rhf_H2.kernel()
e_elec = rhf_H2.energy_elec()

print(f"H2 energy calculated by pyscf = {pyscf_e_H2}")

converged SCF energy = -1.11671432219594
H2 energy calculated by pyscf = -1.1167143221959424


Where we can see that the result is the expected one. Using the implementation of RHF, the electronic energy obtained is: 

In [3]:
from py_mods.src.RHF import RHF
from py_mods.src.scf_utils import V_NN

positions = np.array([
    [0. , 0. , 0. ],
    [0. , 0. , 1.4]
])

nuc_nuc = V_NN(positions, np.array([1,1]), units='Bohr')

# test : SCF convergence for H2 in STO-3G
converged, E_elec, E_e_values, C_munu, P = RHF(S_sto3g_H2, T_sto3g_H2, V_sto3g_H2, eri_sto3g_H2, n_electrons=2, max_iter=100, threshold=1E-14, p_guess='core', verbose=True)


----------------------------------------------------------------------
|   Iter   |           E_iter           |            Delta_e         |
----------------------------------------------------------------------
    0           -2.5055940592310546           -2.5055940592310546
    1           -1.8309999850811094            0.6745940741499452
    2           -1.8309999850811094            0.6745940741499452
Convergence achieved after 2 iterations. Final SCF energy = -1.8309999850811094


And adding the $V_{NN}$ term:


In [4]:
total_energy = E_elec + nuc_nuc
print(f'Total energy = {total_energy}')
print(f'Total_energy - pyscf_reference = {total_energy-pyscf_e_H2}')

Total energy = -1.116714270795395
Total_energy - pyscf_reference = 5.1400547373958716e-08


Which is the same up to a certain numerical aspect, which corresponds to $V_{NN}$ as we will see now with the next example. Since Complex scaling has to be done in atoms due to the non dilation-analicity of the coulomb potential in molecules, let's see the case of He: 

In [5]:
mol_He = gto.M(atom = 'He 0 0 0', spin=0, charge=0, basis='aug-ccpvqz')

T_sto3g_He = mol_He.intor('int1e_kin')
V_sto3g_He = mol_He.intor('int1e_nuc')
S_sto3g_He = mol_He.intor('int1e_ovlp')
eri_sto3g_He = mol_He.intor('int2e')

rhf_He = scf.RHF(mol_He)
# rhf_He.init_guess = 'hcore'
# rhf_He.max_cycle = 0

pyscf_e_He = rhf_He.kernel()
e_elec = rhf_He.energy_elec()

print(f"H2 energy calculated by pyscf = {pyscf_e_He}")

converged SCF energy = -2.86152199563245
H2 energy calculated by pyscf = -2.861521995632449


In [6]:
converged, E_elec, E_e_values, C_munu, P = RHF(S_sto3g_He, T_sto3g_He, V_sto3g_He, eri_sto3g_He, n_electrons=2, max_iter=100, threshold=1E-14, p_guess='core', verbose=True)

----------------------------------------------------------------------
|   Iter   |           E_iter           |            Delta_e         |
----------------------------------------------------------------------
    0           -3.9996224179871307           -3.9996224179871307
    1           -2.7547092304134289            1.2449131875737018
    2           -2.8509905894254222           -0.0962813590119933
    3           -2.8601932014383693           -0.0092026120129471
    4           -2.8613136524750282           -0.0011204510366589
    5           -2.8614841942619509           -0.0001705417869227
    6           -2.8615145672508779           -0.0000303729889271
    7           -2.8615204727233481           -0.0000059054724701
    8           -2.8615216758299633           -0.0000012031066152
    9           -2.8615219274636479           -0.0000002516336846
   10           -2.8615219809542469           -0.0000000534905991
   11           -2.8615219924491764           -0.000000011494

And the difference:

In [7]:
total_energy = E_elec 
print(f'Total energy = {total_energy}')
print(f'Total_energy - pyscf_reference = {total_energy-pyscf_e_He}')

Total energy = -2.861521995632469
Total_energy - pyscf_reference = -1.9984014443252818e-14


Which shows that the source of error in the $H_2$ molecule was the $V_{NN}$ term. 